# Entscheidungsbaum-Analyse des Titanic-Datensatzes


## Einführung

In diesem Tutorial wirst du einen Entscheidungsbaum erstellen, um vorherzusagen, welche Passagiere die Titanic-Katastrophe überlebt haben. Du wirst dabei die wichtigsten Schritte der Datenanalyse durchgehen und einen Entscheidungsbaum trainieren.

Zuerst installierst du die benötigten Bibliotheken:

In [ ]:
%pip install pandas

In [ ]:
%pip install scikit-learn

In [ ]:
%pip install matplotlib

Danach importieren wir die Bibliotheken:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

### Schritt 1: Daten laden und erkunden

Lade zunächst den Titanic-Datensatz:

In [ ]:
# Daten laden
data = pd.read_csv('titanic.csv')

In [ ]:
# Die ersten Zeilen anzeigen
print(data.head())

In [ ]:
# Informationen über den Datensatz
print(data.info())

### Aufgabe 1:
Untersuche die Ausgabe von `data.info()`. Wie viele Spalten und Zeilen hat der Datensatz? Welche Datentypen haben die einzelnen Spalten?

## Schritt 2: Datenvisualisierung

Visualisierungen helfen dir, die Daten besser zu verstehen:

In [ ]:
# Überlebensrate visualisieren
plt.figure(figsize=(8, 6))
survived_counts = data['Survived'].value_counts()
plt.bar(['Nicht überlebt', 'Überlebt'], survived_counts.values)
plt.title('Überlebensrate')
plt.ylabel('Anzahl der Passagiere')
plt.show()

In [ ]:
# Überlebensrate nach Geschlecht
plt.figure(figsize=(8, 6))
data.groupby('Sex')['Survived'].value_counts().unstack().plot(kind='bar', stacked=True)
plt.title('Überlebensrate nach Geschlecht')
plt.xlabel('Geschlecht')
plt.ylabel('Anzahl der Passagiere')
plt.legend(['Nicht überlebt', 'Überlebt'])
plt.show()

### Aufgabe 2:
Erstelle ein weiteres Diagramm, das die Überlebensrate nach Passagierklasse (Pclass) zeigt. Was kannst du aus diesem Diagramm schließen?

**Tipp:** Die Code-Zelle direkt darunter ist absichtlich fast identisch mit der Zelle oben (Überlebensrate nach Geschlecht) aufgebaut. Du musst nur an den markierten Stellen `'Sex'` durch `'Pclass'` ersetzen und die Beschriftungen (Titel, Achse) sinngemäß anpassen.

In [ ]:
# Tipp: Diese Zelle ist fast identisch zur Zelle oben (Überlebensrate nach Geschlecht).
# Du musst nur an den markierten Stellen 'Sex' durch 'Pclass' ersetzen.

plt.figure(figsize=(8, 6))
data.groupby('Sex')['Survived'].value_counts().unstack().plot(kind='bar', stacked=True)  # <- HIER 'Sex' durch 'Pclass' ersetzen
plt.title('Überlebensrate nach Geschlecht')  # <- HIER Titel anpassen, z. B. 'Überlebensrate nach Passagierklasse'
plt.xlabel('Geschlecht')  # <- HIER Achsenbeschriftung anpassen, z. B. 'Passagierklasse'
plt.ylabel('Anzahl der Passagiere')
plt.legend(['Nicht überlebt', 'Überlebt'])
plt.show()

## Schritt 3: Datenvorverarbeitung

Bevor du dein Modell trainierst, musst du die Daten vorbereiten:

In [ ]:
# Unwichtige Spalten entfernen
columns_to_drop = ['Name', 'Ticket', 'Cabin']

# Statt einer List Comprehension:
actual_columns_to_drop = []
for col in columns_to_drop:
    if col in data.columns:
        actual_columns_to_drop.append(col)

data = data.drop(actual_columns_to_drop, axis=1)

# Fehlende Werte behandeln
if 'Age' in data.columns:
    data['Age'] = data['Age'].fillna(data['Age'].median())

if 'Embarked' in data.columns:
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

In [ ]:
# Kategorische Variablen in numerische umwandeln
categorical_columns = ['Sex', 'Embarked']

# Statt einer List Comprehension:
actual_categorical_columns = []
for col in categorical_columns:
    if col in data.columns:
        actual_categorical_columns.append(col)

data = pd.get_dummies(data, columns=actual_categorical_columns, drop_first=True)

print(data.head())
print(f"Spalten nach der Vorverarbeitung: {data.columns.tolist()}")

### Aufgabe 3:
Welche Spalten wurden in deinem Fall entfernt? Warum ist es wichtig, die Existenz von Spalten zu überprüfen, bevor du sie entfernst oder verarbeitest?

## Schritt 4: Daten aufteilen

Teile deine Daten in Trainings- und Testdaten auf. **Wichtig:** Die Testdaten (20 %) legst du jetzt „in den Tresor“. Sie werden erst ganz am Ende **ein einziges Mal** benutzt, um das fertige Modell zu bewerten. Alle Entscheidungen vorher – zum Beispiel welche Baumtiefe du wählst – triffst du nur mit den Trainingsdaten. Würdest du die Tiefe anhand der Testdaten aussuchen, wäre die Testgenauigkeit zu optimistisch, weil das Modell dann indirekt an die Testdaten angepasst wurde.

In [ ]:
X = data.drop('Survived', axis=1)
y = data['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Trainingsdaten Shape: {X_train.shape}")
print(f"Testdaten Shape: {X_test.shape}")

## Schritt 5: Entscheidungsbaum trainieren

Jetzt trainierst du deinen Entscheidungsbaum zunächst ohne Tiefenbegrenzung. Um abzuschätzen, wie gut er auf **unbekannten** Daten funktioniert, ohne die Testdaten anzufassen, verwendest du eine **5-fache Kreuzvalidierung** (`cross_val_score`): Die Trainingsdaten werden in 5 Teile zerlegt; fünfmal wird auf 4 Teilen trainiert und auf dem jeweils fünften Teil (dem Validierungsteil) geprüft. Der Mittelwert der fünf Ergebnisse ist die **Validierungsgenauigkeit**.

Zum Vergleich gibst du auch die **Trainingsgenauigkeit** aus. Ist sie deutlich höher als die Validierungsgenauigkeit, hat sich das Modell die Trainingsdaten „auswendig gemerkt“ (Overfitting).

In [ ]:
# Entscheidungsbaum ohne Tiefenbegrenzung erstellen und trainieren
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Trainingsgenauigkeit: Wie gut passt das Modell zu den Daten, die es schon kennt?
train_accuracy = accuracy_score(y_train, dt.predict(X_train))

# Validierungsgenauigkeit per 5-facher Kreuzvalidierung (nur auf den Trainingsdaten!)
cv_scores = cross_val_score(DecisionTreeClassifier(random_state=42), X_train, y_train, cv=5)

print(f"Trainingsgenauigkeit:     {train_accuracy:.2f}")
print(f"Validierungsgenauigkeit:  {cv_scores.mean():.2f} (± {cv_scores.std():.2f})")

In [ ]:
# Funktion zum Zeichnen des Entscheidungsbaums (Basisversion)
def plot_decision_tree(model, feature_names, class_names, filename=None):
    plt.figure(figsize=(20, 10))
    plot_tree(
        model,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True
    )
    
    # Wenn filename übergeben wurde, das Diagramm als PDF abspeichern
    if filename is not None:
        plt.savefig(filename, format='pdf', bbox_inches='tight')
    
    plt.show()

# Entscheidungsbaum zeichnen und als PDF abspeichern
plot_decision_tree(
    dt,
    feature_names=X.columns,
    class_names=['Nicht überlebt', 'Überlebt'],
    filename='mein_baum.pdf'  # <- beliebiger Dateiname
)

### Aufgabe 4:
Betrachte den gezeichneten Entscheidungsbaum. Welche Features erscheinen nahe der Wurzel des Baums? Was bedeutet das für ihre Wichtigkeit bei der Vorhersage? Versuche, den Entscheidungsprozess für einen Passagier nachzuvollziehen, indem du den Baum von der Wurzel zu einem Blatt verfolgst.

Jetzt erstellst du eine Funktion, die es dir ermöglicht, die Tiefe des Entscheidungsbaums anzupassen und die Auswirkungen zu beobachten. Sie gibt jeweils Trainings- und Validierungsgenauigkeit aus – die Testdaten bleiben weiterhin unberührt.

In [ ]:
def train_visualize_evaluate_tree(X_train, y_train, max_depth):
    # Entscheidungsbaum erstellen und trainieren
    dt = DecisionTreeClassifier(random_state=42, max_depth=max_depth)
    dt.fit(X_train, y_train)

    # Trainings- und Validierungsgenauigkeit berechnen
    train_acc = accuracy_score(y_train, dt.predict(X_train))
    cv_acc = cross_val_score(DecisionTreeClassifier(random_state=42, max_depth=max_depth),
                             X_train, y_train, cv=5).mean()
    print(f"Tiefe {max_depth}: Trainingsgenauigkeit {train_acc:.2f}, Validierungsgenauigkeit {cv_acc:.2f}")

    # Entscheidungsbaum zeichnen
    plt.figure(figsize=(20, 10))
    plot_tree(
        dt,
        feature_names=X_train.columns,
        class_names=['Nicht überlebt', 'Überlebt'],
        filled=True,
        rounded=True,
        fontsize=10
    )
    plt.title(f"Entscheidungsbaum mit max_depth={max_depth}")
    plt.show()

    return dt, train_acc, cv_acc

In [ ]:
# Versuche verschiedene Tiefen
depths_to_try = [1, 3, 5, 10]
results = {}

for depth in depths_to_try:
    print(f"\nTrainiere Baum mit max_depth = {depth}")
    tree, train_acc, cv_acc = train_visualize_evaluate_tree(X_train, y_train, depth)
    results[depth] = (tree, train_acc, cv_acc)

Die folgende Zelle berechnet Trainings- und Validierungsgenauigkeit für alle Tiefen von 1 bis 12 (ohne Baumzeichnung) und stellt beide Kurven gegenüber. So erkennst du, ab welcher Tiefe sich die beiden Kurven auseinanderentwickeln.

In [ ]:
tiefen = range(1, 13)
train_werte = []
cv_werte = []

for tiefe in tiefen:
    modell = DecisionTreeClassifier(random_state=42, max_depth=tiefe).fit(X_train, y_train)
    train_werte.append(accuracy_score(y_train, modell.predict(X_train)))
    cv_werte.append(cross_val_score(DecisionTreeClassifier(random_state=42, max_depth=tiefe),
                                    X_train, y_train, cv=5).mean())
    print(f"Tiefe {tiefe:2d}: Training {train_werte[-1]:.3f}  Validierung {cv_werte[-1]:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(list(tiefen), train_werte, marker='o', label='Trainingsgenauigkeit')
plt.plot(list(tiefen), cv_werte, marker='o', label='Validierungsgenauigkeit (5-fache KV)')
plt.xlabel('max_depth')
plt.ylabel('Genauigkeit')
plt.title('Trainings- und Validierungsgenauigkeit je Baumtiefe')
plt.legend()
plt.grid(True)
plt.show()

### Aufgabe 5:
Betrachte die Entscheidungsbäume sowie die Trainings- und Validierungsgenauigkeiten für verschiedene Tiefen.

a) Wie verändert sich die Struktur des Baumes mit zunehmender Tiefe?<br>
b) Wie wirken sich die Tiefe auf die Trainings- und auf die Validierungsgenauigkeit aus?<br>
c) Ab welcher Tiefe beginnt das Modell zu überanpassen (Overfitting)? Woran erkennst du das?

### Aufgabe 6:
Wähle die Baumtiefe aus, die deiner Meinung nach den besten Kompromiss zwischen Validierungsgenauigkeit und Interpretierbarkeit bietet. Begründe deine Wahl. Die Zelle unten schlägt die Tiefe mit der höchsten Validierungsgenauigkeit vor – du kannst die markierte Zeile mit deiner eigenen Wahl überschreiben.

In [ ]:
# Vorschlag: Tiefe mit der höchsten Validierungsgenauigkeit
beste_tiefe = list(tiefen)[int(np.argmax(cv_werte))]
print(f"Höchste Validierungsgenauigkeit bei Tiefe {beste_tiefe}: {max(cv_werte):.3f}")

gewaehlte_tiefe = beste_tiefe  # <-- HIER KANNST DU ÄNDERN, z. B. gewaehlte_tiefe = 3

## Schritt 6: Modellauswertung

Erst jetzt, nachdem die Tiefe feststeht, trainierst du das endgültige Modell auf allen Trainingsdaten und bewertest es **einmal** auf den Testdaten. Konfusionsmatrix und Feature Importance beziehen sich auf genau dieses gewählte Modell.

In [ ]:
# Endgültiges Modell mit der gewählten Tiefe
dt_final = DecisionTreeClassifier(random_state=42, max_depth=gewaehlte_tiefe)
dt_final.fit(X_train, y_train)

y_pred_final = dt_final.predict(X_test)

print(f"Gewählte Tiefe:          {gewaehlte_tiefe}")
print(f"Trainingsgenauigkeit:    {accuracy_score(y_train, dt_final.predict(X_train)):.2f}")
print(f"Testgenauigkeit:         {accuracy_score(y_test, y_pred_final):.2f}")

In [ ]:
# Konfusionsmatrix des gewählten Modells
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title(f'Konfusionsmatrix (max_depth={gewaehlte_tiefe})')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Nicht überlebt', 'Überlebt'])
plt.yticks(tick_marks, ['Nicht überlebt', 'Überlebt'])
plt.ylabel('Tatsächliche Werte')
plt.xlabel('Vorhergesagte Werte')
plt.show()

print(cm)

In [ ]:
# Feature Importance des gewählten Modells
importances = dt_final.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

for i in indices:
    print(f"{feature_names[i]:28s} {importances[i]:.3f}")

plt.figure(figsize=(10, 6))
plt.title(f"Feature Importances (max_depth={gewaehlte_tiefe})")
plt.bar(range(X.shape[1]), importances[indices])
plt.xticks(range(X.shape[1]), [feature_names[i] for i in indices], rotation=90)
plt.tight_layout()
plt.show()

### Aufgabe 7:
Interpretiere die Konfusionsmatrix. Was sagen dir die Zahlen in der Matrix? Welche Features scheinen laut der Feature Importance am wichtigsten für die Vorhersage zu sein?

## Zusammenfassung

Der Ablauf zur Datenanalyse mittels Entscheidungsbaum umfasst folgende Schritte:
1. Daten laden und erkunden
2. Daten visualisieren
3. Daten vorverarbeiten
4. Daten in Trainings- und Testdaten aufteilen (Testdaten erst am Ende verwenden)
5. Entscheidungsbäume trainieren und die Tiefe per Kreuzvalidierung auf den Trainingsdaten wählen
6. Das gewählte Modell einmal auf den Testdaten auswerten